In [13]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum as spark_sum, avg, count, rank, row_number, when
from pyspark.sql.window import Window

spark = SparkSession.builder \
    .appName("Pertemuan5-JoinWindowSQL") \
    .master("local[*]") \
    .getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

print("SparkSession siap. Versi Spark:", spark.version)

SparkSession siap. Versi Spark: 3.5.9


In [15]:
import numpy as np
import pandas as pd

np.random.seed(7)
kategori_list = ["Elektronik", "Fashion", "Makanan & Minuman", "Kesehatan & Kecantikan", "Rumah Tangga"]

# Tabel referensi/master: target & manager per kategori (data ini relatif statis, jarang berubah)
data_produk = {
    "kategori": kategori_list,
    "target_bulanan": [50000000, 40000000, 30000000, 25000000, 20000000],
    "manager": ["Andi", "Budi", "Citra", "Dewi", "Eka"],
}
df_produk = spark.createDataFrame(pd.DataFrame(data_produk))

# Tabel transaksi: data yang terus bertambah setiap hari
n = 300
data_transaksi = {
    "order_id": [f"O{i}" for i in range(n)],
    "kategori": np.random.choice(kategori_list, size=n),
    "kota": np.random.choice(["Magelang", "Semarang", "Solo"], size=n),
    "pendapatan": np.random.randint(50000, 500000, size=n),
}
df_transaksi = spark.createDataFrame(pd.DataFrame(data_transaksi))

print("df_produk:")
df_produk.show()
print("df_transaksi (5 baris pertama dari total", df_transaksi.count(), "baris):")
df_transaksi.show(5)

df_produk:


+--------------------+--------------+-------+
|            kategori|target_bulanan|manager|
+--------------------+--------------+-------+
|          Elektronik|      50000000|   Andi|
|             Fashion|      40000000|   Budi|
|   Makanan & Minuman|      30000000|  Citra|
|Kesehatan & Kecan...|      25000000|   Dewi|
|        Rumah Tangga|      20000000|    Eka|
+--------------------+--------------+-------+



df_transaksi (5 baris pertama dari total 300 baris):
+--------+--------------------+--------+----------+
|order_id|            kategori|    kota|pendapatan|
+--------+--------------------+--------+----------+
|      O0|        Rumah Tangga|Magelang|    488643|
|      O1|             Fashion|Magelang|    401943|
|      O2|Kesehatan & Kecan...|    Solo|    452308|
|      O3|Kesehatan & Kecan...|Semarang|    421741|
|      O4|        Rumah Tangga|Semarang|    185244|
+--------+--------------------+--------+----------+
only showing top 5 rows



In [16]:
df_gabung = df_transaksi.join(df_produk, on="kategori", how="left")
df_gabung.show(5)

+--------------------+--------+--------+----------+--------------+-------+
|            kategori|order_id|    kota|pendapatan|target_bulanan|manager|
+--------------------+--------+--------+----------+--------------+-------+
|Kesehatan & Kecan...|      O2|    Solo|    452308|      25000000|   Dewi|
|Kesehatan & Kecan...|      O3|Semarang|    421741|      25000000|   Dewi|
|             Fashion|      O1|Magelang|    401943|      40000000|   Budi|
|             Fashion|      O5|Semarang|    155828|      40000000|   Budi|
|        Rumah Tangga|      O0|Magelang|    488643|      20000000|    Eka|
+--------------------+--------+--------+----------+--------------+-------+
only showing top 5 rows



In [17]:
# Langkah 1: Meringkas total pendapatan per kategori
ringkasan = df_transaksi.groupBy("kategori").agg(
    spark_sum("pendapatan").alias("total_pendapatan")
)

# Langkah 2: Join dengan tabel target untuk menghitung pencapaian
hasil = ringkasan.join(df_produk, on="kategori", how="inner")
hasil = hasil.withColumn(
    "pencapaian_persen",
    (col("total_pendapatan") / col("target_bulanan") * 100)
)

hasil.orderBy(col("pencapaian_persen").desc()).show()

[Stage 14:=================================>                       (7 + 5) / 12]

+--------------------+----------------+--------------+-------+------------------+
|            kategori|total_pendapatan|target_bulanan|manager| pencapaian_persen|
+--------------------+----------------+--------------+-------+------------------+
|        Rumah Tangga|        17243718|      20000000|    Eka| 86.21858999999999|
|Kesehatan & Kecan...|        17607693|      25000000|   Dewi| 70.43077199999999|
|   Makanan & Minuman|        13338394|      30000000|  Citra| 44.46131333333334|
|             Fashion|        16588742|      40000000|   Budi|41.471855000000005|
|          Elektronik|        16300808|      50000000|   Andi|         32.601616|
+--------------------+----------------+--------------+-------+------------------+



In [20]:
df_right = df_transaksi.join(df_produk, on="kategori", how="right")
df_right.show(5)

[Stage 31:=================================>                       (7 + 5) / 12]

+----------+--------+--------+----------+--------------+-------+
|  kategori|order_id|    kota|pendapatan|target_bulanan|manager|
+----------+--------+--------+----------+--------------+-------+
|Elektronik|    O295|Semarang|    321602|      50000000|   Andi|
|Elektronik|    O294|    Solo|    285847|      50000000|   Andi|
|Elektronik|    O293|Magelang|    351373|      50000000|   Andi|
|Elektronik|    O285|Semarang|    399142|      50000000|   Andi|
|Elektronik|    O278|Semarang|    350060|      50000000|   Andi|
+----------+--------+--------+----------+--------------+-------+
only showing top 5 rows



In [21]:
df_outer = df_transaksi.join(df_produk, on="kategori", how="outer")
df_outer.show(5)

[Stage 37:=========>                                              (2 + 10) / 12]

+----------+--------+--------+----------+--------------+-------+
|  kategori|order_id|    kota|pendapatan|target_bulanan|manager|
+----------+--------+--------+----------+--------------+-------+
|Elektronik|      O6|Semarang|    491344|      50000000|   Andi|
|Elektronik|     O10|    Solo|     72294|      50000000|   Andi|
|Elektronik|     O12|    Solo|     75566|      50000000|   Andi|
|Elektronik|     O14|    Solo|    459719|      50000000|   Andi|
|Elektronik|     O20|Magelang|    363163|      50000000|   Andi|
+----------+--------+--------+----------+--------------+-------+
only showing top 5 rows



In [6]:
# Mendefinisikan "window": kelompokkan berdasarkan kategori, urutkan dari pendapatan tertinggi
window_spec = Window.partitionBy("kategori").orderBy(col("pendapatan").desc())

# Menambahkan kolom peringkat SETIAP transaksi di dalam kategorinya masing-masing
df_ranked = df_transaksi.withColumn("rank_dalam_kategori", rank().over(window_spec))

# Menampilkan hanya 2 transaksi teratas (rank 1 dan 2) di setiap kategori
df_ranked.filter(col("rank_dalam_kategori") <= 2) \
    .orderBy("kategori", "rank_dalam_kategori") \
    .show(10)

[Stage 19:>                                                       (0 + 12) / 12]

+--------+--------------------+--------+----------+-------------------+
|order_id|            kategori|    kota|pendapatan|rank_dalam_kategori|
+--------+--------------------+--------+----------+-------------------+
|    O197|          Elektronik|Magelang|    495470|                  1|
|      O6|          Elektronik|Semarang|    491344|                  2|
|     O96|             Fashion|Magelang|    498770|                  1|
|     O42|             Fashion|Semarang|    492607|                  2|
|     O50|Kesehatan & Kecan...|Magelang|    497864|                  1|
|     O72|Kesehatan & Kecan...|    Solo|    478508|                  2|
|    O215|   Makanan & Minuman|Magelang|    493687|                  1|
|    O190|   Makanan & Minuman|Semarang|    470049|                  2|
|    O286|        Rumah Tangga|    Solo|    497826|                  1|
|    O147|        Rumah Tangga|Semarang|    497370|                  2|
+--------+--------------------+--------+----------+-------------

In [7]:
df_transaksi.createOrReplaceTempView("transaksi")
df_produk.createOrReplaceTempView("produk")

print("Kedua tabel sementara berhasil didaftarkan: 'transaksi' dan 'produk'")

Kedua tabel sementara berhasil didaftarkan: 'transaksi' dan 'produk'


In [8]:
hasil_sql = spark.sql('''
    SELECT t.kategori, p.manager, SUM(t.pendapatan) AS total_pendapatan
    FROM transaksi t
    JOIN produk p ON t.kategori = p.kategori
    GROUP BY t.kategori, p.manager
    ORDER BY total_pendapatan DESC
''')
hasil_sql.show()

[Stage 23:==========================================>              (9 + 3) / 12]

+--------------------+-------+----------------+
|            kategori|manager|total_pendapatan|
+--------------------+-------+----------------+
|Kesehatan & Kecan...|   Dewi|        17607693|
|        Rumah Tangga|    Eka|        17243718|
|             Fashion|   Budi|        16588742|
|          Elektronik|   Andi|        16300808|
|   Makanan & Minuman|  Citra|        13338394|
+--------------------+-------+----------------+



In [9]:
spark.stop()
print("SparkSession ditutup.")

SparkSession ditutup.
